# **LightGBM·CatBoost GPU 비교**

같은 판매 이력과 시간 분할에서 두 트리 모델을 학습해 **7일 수요 계획에 더 도움이 되는 모델**을 고릅니다. LightGBM과 CatBoost는 기준일 `t`의 정보로 `t+1`~`t+7`의 일별 판매를 예측합니다.

이 노트북은 Colab의 저장소에서 전체 학습을 실행하고 Valid/Test 예측을 저장합니다. 학습은 오래 걸리므로 설정과 기존 결과를 확인한 뒤 필요한 셀만 실행합니다.

## **환경 준비**

별도의 00 노트북에서 저장소를 My Drive에 clone·갱신한 뒤 실행합니다. Drive를 연결하고 `/content/drive/MyDrive/retail-demand-forecasting`을 작업 폴더로 사용합니다.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive/")
except ImportError:
    pass

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

### **패키지 설치**

00 노트북에서 준비한 저장소의 의존성을 설치합니다. LightGBM은 사전 빌드된 OpenCL GPU 패키지를 사용하므로 CUDA 컴파일러가 필요하지 않습니다.


In [ ]:
%pip install -r requirements.txt

%pip install -q -e . --no-deps

## **Library**


In [ ]:
import gc
import json
from dataclasses import replace

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

from retail_forecast.config import load_config
from retail_forecast.metrics import regression_metrics
from retail_forecast.pipeline import feature_cache_path, run_time_split_experiment

## **Utils**

실험 설정과 결과 표를 위한 보조 함수만 이곳에 둡니다. Feature 생성과 모델 학습은 `src/retail_forecast`의 함수를 사용합니다.


In [ ]:
# 노트북 설정을 기존 TOML 설정에 반영합니다.
def experiment_config(path, overrides, run_name):
    config = load_config(path)
    return replace(
        config,
        model_params={**config.model_params, **overrides},
        model_path=config.model_path.parent / run_name,
        output_path=config.output_path.parent / run_name,
    )

In [ ]:
# 저장된 예측과 지표를 모델별 비교표로 묶습니다.
def build_comparison_tables(result_dir):
    rows, horizons = [], []
    for model_name in ("lightgbm", "catboost"):
        metadata = json.loads((result_dir / f"metrics_{model_name}.json").read_text(encoding="utf-8"))
        predictions = pd.read_csv(result_dir / f"test_predictions_{model_name}.csv")
        decisions = pd.read_csv(result_dir / f"test_decision_summary_{model_name}.csv")
        weekly = regression_metrics(decisions["actual_sum_1_7"], decisions["prediction_sum_1_7"])
        rows.append({
            "model": model_name,
            "selected_iterations": metadata["selected_iterations"],
            "series_count": metadata["series_count"],
            **metadata["test"],
            **{f"sum_1_7_{key}": value for key, value in weekly.items()},
        })
        for horizon, group in predictions.groupby("forecast_horizon"):
            horizons.append({
                "model": model_name, "forecast_horizon": horizon, "rows": len(group),
                **regression_metrics(group["target_sales"], group["prediction"]),
            })
    return pd.DataFrame(rows), pd.DataFrame(horizons)

## **실험 파라미터**

1200은 트리 깊이가 아니라 최대 생성 횟수입니다. 두 모델의 최대 횟수를 우선 600으로 제한하고 조기 종료를 유지합니다. LightGBM `max_depth=-1`은 깊이 제한 없음이며, CatBoost는 `depth=8`입니다. 다른 파라미터도 아래 모델별 딕셔너리에 추가하면 TOML 기본값을 덮어씁니다. 설정을 바꿔 다시 학습할 때는 `RUN_NAME`도 변경해 기존 결과를 보존하세요.


In [ ]:
sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

RUN_NAME = "full_history_time_split_fixed_i600"
MODEL_OVERRIDES = {
    "lightgbm": {"n_estimators": 600, "max_depth": -1, "early_stopping_rounds": 100},
    "catboost": {"iterations": 600, "depth": 8, "gpu_ram_part": 0.8, "early_stopping_rounds": 100},
}


## **비교 모델**

| 모델 | 학습 방식 |
|---|---|
| LightGBM | 전체 이력 Feature 기반 GPU 학습 |
| CatBoost | 같은 Feature·검증 기준일의 GPU 학습 |

딥러닝 N-HiTS·TCN·TFT는 [07 Colab 노트북](07_dl_colab_validation.ipynb)에서 별도로 실행합니다.


## **1. Kaggle 원본과 전체 이력 Feature**

Feature 캐시가 이미 있으면 이 셀은 건너뛰세요. 최초 생성 시 `train.csv`, `stores.csv`, `holidays_events.csv`, `transactions.csv`를 저장소의 `data/raw`에 넣습니다. 스크립트가 28일 단위 Parquet으로 저장하며, 완성된 캐시가 있으면 재생성하지 않습니다.


In [ ]:
!python scripts/build_full_features.py --config configs/full_lightgbm_gpu.toml --chunk-days 28


## **2. 모델별 전체 이력 학습**

LightGBM은 L2(`regression`), CatBoost는 제곱오차 계열(`RMSE`)로 학습합니다. 두 설정은 같은 Feature 캐시와 검증 구간을 사용합니다. LightGBM 다음 CatBoost를 실행하세요. 결과는 `models/{RUN_NAME}`, `outputs/{RUN_NAME}`에 저장됩니다. 기존 결과를 보존하려면 실행 전에 `RUN_NAME`을 바꾸세요.

In [ ]:
lightgbm_config = experiment_config(
    REPO / "configs/full_lightgbm_gpu.toml", MODEL_OVERRIDES["lightgbm"], RUN_NAME,
)
if not feature_cache_path(lightgbm_config).exists():
    raise FileNotFoundError("전체 이력 Feature 캐시가 없습니다. 이전 셀을 먼저 실행하세요.")
lightgbm_result = run_time_split_experiment(lightgbm_config, progress=True)
lightgbm_result

In [ ]:
gc.collect()
catboost_config = experiment_config(
    REPO / "configs/full_catboost_gpu.toml", MODEL_OVERRIDES["catboost"], RUN_NAME,
)
catboost_result = run_time_split_experiment(catboost_config, progress=True)
catboost_result

## **3. 결과 표 저장**

전체 일별 예측, horizon별 예측, 발주용 7일 합계의 오차를 별도 표로 저장합니다. 두 모델의 결과 파일이 모두 있을 때만 비교표를 만듭니다.

### **모델별 Test 종합 지표**


In [ ]:
result_dir = lightgbm_config.output_path
comparison, by_horizon = build_comparison_tables(result_dir)
comparison.to_csv(result_dir / "comparison_time_split_test.csv", index=False)
by_horizon.to_csv(result_dir / "comparison_by_horizon_time_split_test.csv", index=False)
comparison

### **예측 거리별 Test 지표**


In [ ]:
by_horizon

## **모델 비교 시각화**


In [ ]:
plot_data = comparison[["model", "rmsle", "mae", "rmse", "wape"]]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric in zip(axes, ("rmsle", "mae", "rmse", "wape")):
    sns.barplot(data=plot_data, x=metric, y="model", ax=ax, color="steelblue")
    ax.set(title=metric.upper(), xlabel="낮을수록 좋음", ylabel="")
fig.tight_layout()
plt.show()

## **모델 선택과 해석**

WAPE와 7일 합계 오차를 중심으로 MAE, RMSE, RMSLE, Bias도 함께 봅니다. WAPE는 개별 판매량이 0이어도 계산할 수 있지만, 같은 검증 행에서는 MAE와 모델 순위가 같습니다. 성능 차이가 작다면 학습 시간과 설명 편의성을 고려합니다. 상품군·매장별 오류는 `06_error_analysis.ipynb`에서 분석합니다.

딥러닝 모델과의 최종 공통 행 비교는 저장된 예측을 읽는 [08 평가](08_model_evaluation_and_decision.ipynb)에서 확인합니다.


### **해석과 주의사항**

- 전체 이력은 학습에 허용된 과거 기준일을 넓게 사용한다는 뜻입니다. Valid로 종료 시점을 선택한 모델을 고정하고 Test를 평가하므로 Test 정답으로 다시 학습하지 않습니다.
- 이전 365일 결과와 학습 기간과 Feature 조건이 달라 숫자를 직접 비교하면 안 됩니다. 이번 설정은 예측 시점에 알 수 없는 지진·돌발 이벤트 변수를 제외합니다.
- 원본 `train.csv`의 `target_onpromotion`은 미래의 실제 판촉 수입니다. 현재 Feature 표에는 남아 있지만 `get_feature_columns()`에서 제외하므로 LightGBM과 CatBoost의 입력으로 사용하지 않습니다.
- `sales`와 `current_transactions`는 기준일 영업 종료 후 예측한다는 전제입니다. 영업 중 발주라면 이 변수도 제외해야 합니다.
- 전체 이력 표는 메모리를 많이 사용합니다. Colab의 GPU와 RAM 할당은 변할 수 있고, My Drive I/O는 느릴 수 있습니다. 런타임이 끊겨도 완성된 Parquet과 결과 파일은 Drive에 남습니다. `.building` 폴더가 남았다면 생성이 중단된 것이므로 내용 확인 후 수동으로 정리하세요.